# PuttingCupintotheDish Demo Visualization

`/media/hyunjin/T7/rby1_demo/PuttingCupintotheDish` 경로의 데모 데이터 구조 탐색 및 비디오 시각화

In [ ]:
import os
import glob
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

DATASET_ROOT = Path("/media/hyunjin/T7/rby1_demo/PuttingCupintotheDish")
print(f"Dataset root: {DATASET_ROOT}")
print(f"Dataset exists: {DATASET_ROOT.exists()}")

## 1. Dataset 전체 구조 탐색

In [ ]:
# 에피소드 목록 수집
episode_dirs = sorted(DATASET_ROOT.glob("episode_*"), key=lambda x: int(x.name.split("_")[1]))
print(f"총 에피소드 수: {len(episode_dirs)}")
print(f"에피소드 목록: {[d.name for d in episode_dirs]}\n")

# 각 에피소드의 파일 목록 및 HDF5 구조 요약
summary = []
for ep_dir in episode_dirs:
    files = list(ep_dir.iterdir())
    h5_files = [f for f in files if f.suffix == '.h5']
    other_files = [f for f in files if f.suffix != '.h5']
    
    info = {"episode": ep_dir.name, "h5_files": [f.name for f in h5_files], "other": [f.name for f in other_files]}
    
    # HDF5 파일에서 메타정보 읽기
    if h5_files:
        with h5py.File(h5_files[0], 'r') as f:
            groups = list(f.keys())
            # samples 그룹에서 타임스텝 수 확인
            n_steps = f['samples/time'].shape[0] if 'samples/time' in f else 0
            n_frames = f['head_rgb/image'].shape[0] if 'head_rgb/image' in f else 0
            info["n_steps"] = n_steps
            info["n_frames"] = n_frames
            info["groups"] = groups
    
    summary.append(info)
    print(f"[{ep_dir.name}] files={[f.name for f in files]}, n_steps={info.get('n_steps')}, n_frames(rgb)={info.get('n_frames')}")

In [ ]:
# 에피소드별 n_steps, n_frames 분포 시각화
n_steps_list = [s['n_steps'] for s in summary]
n_frames_list = [s['n_frames'] for s in summary]
ep_indices = [int(s['episode'].split('_')[1]) for s in summary]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(ep_indices, n_steps_list, color='steelblue', alpha=0.8)
axes[0].set_title("에피소드별 Robot State Steps 수", fontsize=13)
axes[0].set_xlabel("Episode Index")
axes[0].set_ylabel("Timesteps")
axes[0].axhline(np.mean(n_steps_list), color='red', linestyle='--', label=f"평균: {np.mean(n_steps_list):.1f}")
axes[0].legend()

axes[1].bar(ep_indices, n_frames_list, color='coral', alpha=0.8)
axes[1].set_title("에피소드별 RGB 프레임 수 (head_rgb)", fontsize=13)
axes[1].set_xlabel("Episode Index")
axes[1].set_ylabel("Frames")
axes[1].axhline(np.mean(n_frames_list), color='red', linestyle='--', label=f"평균: {np.mean(n_frames_list):.1f}")
axes[1].legend()

plt.tight_layout()
plt.suptitle(f"PuttingCupintotheDish Dataset Overview (총 {len(summary)}개 에피소드)", y=1.02, fontsize=14, fontweight='bold')
plt.show()

print(f"\n[Steps 통계] min={min(n_steps_list)}, max={max(n_steps_list)}, mean={np.mean(n_steps_list):.1f}")
print(f"[Frames 통계] min={min(n_frames_list)}, max={max(n_frames_list)}, mean={np.mean(n_frames_list):.1f}")

## 2. 단일 에피소드 HDF5 상세 구조 확인

In [ ]:
EPISODE_IDX = 0  # 확인할 에피소드 번호 (변경 가능)

ep_dir = DATASET_ROOT / f"episode_{EPISODE_IDX}"
h5_path = list(ep_dir.glob("*.h5"))[0]
print(f"파일: {h5_path}\n")

def print_h5_structure(name, obj):
    indent = "  " * name.count("/")
    if isinstance(obj, h5py.Dataset):
        print(f"{indent}📊 [{name}]  shape={obj.shape}  dtype={obj.dtype}")
    elif isinstance(obj, h5py.Group):
        print(f"{indent}📁 [{name}]")

with h5py.File(h5_path, 'r') as f:
    print("=== HDF5 File Structure ===")
    f.visititems(print_h5_structure)
    
    print("\n=== samples 키 목록 ===")
    if 'samples' in f:
        for key in f['samples'].keys():
            ds = f[f'samples/{key}']
            print(f"  samples/{key}: shape={ds.shape}, dtype={ds.dtype}")

## 3. RGB 카메라 영상 프레임 시각화 (head / left / right)

In [ ]:
N_SAMPLE_FRAMES = 6  # 시각화할 프레임 수

cameras = ['head_rgb', 'left_rgb', 'right_rgb']
cam_labels = {'head_rgb': 'Head Camera', 'left_rgb': 'Left Camera', 'right_rgb': 'Right Camera'}

with h5py.File(h5_path, 'r') as f:
    # 각 카메라에서 균등하게 N_SAMPLE_FRAMES개 프레임 추출
    cam_frames = {}
    for cam in cameras:
        if f'{cam}/image' in f:
            images = f[f'{cam}/image'][:]  # (T, H, W, 3)
            total = images.shape[0]
            indices = np.linspace(0, total - 1, N_SAMPLE_FRAMES, dtype=int)
            cam_frames[cam] = (images[indices], indices, total)

fig, axes = plt.subplots(len(cam_frames), N_SAMPLE_FRAMES, figsize=(N_SAMPLE_FRAMES * 3, len(cam_frames) * 2.5))
fig.suptitle(f"Episode {EPISODE_IDX} — RGB Camera Frames", fontsize=15, fontweight='bold')

for row, cam in enumerate(cameras):
    if cam not in cam_frames:
        continue
    images, indices, total = cam_frames[cam]
    for col in range(N_SAMPLE_FRAMES):
        ax = axes[row][col]
        ax.imshow(images[col])
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(cam_labels[cam], fontsize=11, rotation=90, labelpad=10)
            ax.yaxis.set_label_coords(-0.15, 0.5)
        ax.set_title(f"t={indices[col]}/{total-1}", fontsize=8)

plt.tight_layout()
plt.show()

## 4. Depth 이미지 시각화 (head_depth)

In [ ]:
with h5py.File(h5_path, 'r') as f:
    if 'head_depth/image' in f:
        depth_images = f['head_depth/image'][:]  # (T, H, W) uint16
        total = depth_images.shape[0]
        indices = np.linspace(0, total - 1, N_SAMPLE_FRAMES, dtype=int)
        sampled_depth = depth_images[indices]
    else:
        sampled_depth = None
        print("head_depth/image 데이터 없음")

if sampled_depth is not None:
    fig, axes = plt.subplots(1, N_SAMPLE_FRAMES, figsize=(N_SAMPLE_FRAMES * 3, 2.8))
    fig.suptitle(f"Episode {EPISODE_IDX} — Head Depth Frames", fontsize=14, fontweight='bold')
    
    for col in range(N_SAMPLE_FRAMES):
        ax = axes[col]
        depth_frame = sampled_depth[col].astype(np.float32)
        # 유효 depth 범위 클리핑 (0~5000mm 기준)
        depth_frame = np.clip(depth_frame, 0, 5000)
        im = ax.imshow(depth_frame, cmap='plasma', vmin=0, vmax=5000)
        ax.axis('off')
        ax.set_title(f"t={indices[col]}", fontsize=9)
    
    plt.colorbar(im, ax=axes[-1], label="Depth (mm)", shrink=0.8)
    plt.tight_layout()
    plt.show()
    print(f"Depth 통계: min={depth_images.min()}, max={depth_images.max()}, mean={depth_images.mean():.1f}")

## 5. Robot State (관절각도, 그리퍼) 시계열 시각화

In [ ]:
with h5py.File(h5_path, 'r') as f:
    time = f['samples/time'][:]
    robot_pos = f['samples/robot_position'][:]       # (T, 24)
    robot_target = f['samples/robot_target_joints'][:]  # (T, 24)
    gripper_state = f['samples/gripper_state'][:]    # (T, 2)
    gripper_target = f['samples/gripper_target'][:] # (T, 2)
    base_state = f['samples/base_state'][:]          # (T, 3)

time_rel = time - time[0]  # 시작 시간 기준 상대 시간

fig, axes = plt.subplots(3, 1, figsize=(14, 11))
fig.suptitle(f"Episode {EPISODE_IDX} — Robot State Time Series", fontsize=14, fontweight='bold')

# 관절각도 (robot_position)
ax = axes[0]
n_joints = robot_pos.shape[1]
cmap = plt.cm.get_cmap('tab20', n_joints)
for j in range(n_joints):
    ax.plot(time_rel, np.degrees(robot_pos[:, j]), color=cmap(j), linewidth=1.2, alpha=0.8, label=f"J{j}")
ax.set_title("Robot Joint Positions (deg)", fontsize=12)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Angle (°)")
ax.legend(loc='upper right', ncol=6, fontsize=7, framealpha=0.7)
ax.grid(True, alpha=0.3)

# 그리퍼 상태
ax = axes[1]
ax.plot(time_rel, gripper_state[:, 0], 'b-', linewidth=2, label="Left Gripper State")
ax.plot(time_rel, gripper_state[:, 1], 'r-', linewidth=2, label="Right Gripper State")
ax.plot(time_rel, gripper_target[:, 0], 'b--', linewidth=1.5, alpha=0.7, label="Left Gripper Target")
ax.plot(time_rel, gripper_target[:, 1], 'r--', linewidth=1.5, alpha=0.7, label="Right Gripper Target")
ax.set_title("Gripper State & Target", fontsize=12)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Gripper Value")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Base state (x, y, yaw)
ax = axes[2]
base_labels = ['x (m)', 'y (m)', 'yaw (rad)']
colors = ['steelblue', 'coral', 'green']
for i, (lbl, col) in enumerate(zip(base_labels, colors)):
    ax.plot(time_rel, base_state[:, i], color=col, linewidth=2, label=lbl)
ax.set_title("Base State (x, y, yaw)", fontsize=12)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Value")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"총 타임스텝: {len(time_rel)}, 시간 범위: {time_rel[0]:.2f}s ~ {time_rel[-1]:.2f}s (지속시간: {time_rel[-1]:.2f}s)")

## 6. 비디오 생성 및 재생 (head / left / right RGB)

In [ ]:
import matplotlib.animation as animation
from IPython.display import HTML

VIDEO_EPISODE_IDX = 0  # 비디오로 만들 에피소드 번호 (변경 가능)
VIDEO_CAMERA = 'head_rgb'  # 'head_rgb', 'left_rgb', 'right_rgb' 중 선택
VIDEO_FPS = 10             # 재생 FPS

ep_dir_v = DATASET_ROOT / f"episode_{VIDEO_EPISODE_IDX}"
h5_path_v = list(ep_dir_v.glob("*.h5"))[0]

with h5py.File(h5_path_v, 'r') as f:
    frames = f[f'{VIDEO_CAMERA}/image'][:]  # (T, H, W, 3)

print(f"에피소드 {VIDEO_EPISODE_IDX}, 카메라: {VIDEO_CAMERA}")
print(f"총 {frames.shape[0]}프레임, 해상도: {frames.shape[2]}x{frames.shape[1]}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.axis('off')
ax.set_title(f"Episode {VIDEO_EPISODE_IDX} — {VIDEO_CAMERA}", fontsize=12, fontweight='bold')

im = ax.imshow(frames[0])
frame_text = ax.text(5, 15, '', color='white', fontsize=11, fontweight='bold',
                     bbox=dict(facecolor='black', alpha=0.5, boxstyle='round,pad=0.2'))

def update(i):
    im.set_data(frames[i])
    frame_text.set_text(f"Frame {i}/{len(frames)-1}")
    return [im, frame_text]

ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=1000 // VIDEO_FPS, blit=True)
plt.close()

HTML(ani.to_jshtml())

## 7. 3-카메라 동시 재생 (head / left / right 나란히)

In [ ]:
MULTI_EPISODE_IDX = 0  # 에피소드 번호 (변경 가능)
MULTI_FPS = 10

ep_dir_m = DATASET_ROOT / f"episode_{MULTI_EPISODE_IDX}"
h5_path_m = list(ep_dir_m.glob("*.h5"))[0]

with h5py.File(h5_path_m, 'r') as f:
    all_frames = {}
    for cam in ['head_rgb', 'left_rgb', 'right_rgb']:
        if f'{cam}/image' in f:
            all_frames[cam] = f[f'{cam}/image'][:]
        else:
            all_frames[cam] = None

# 유효 카메라만 추출
valid_cams = [(cam, all_frames[cam]) for cam in ['head_rgb', 'left_rgb', 'right_rgb'] if all_frames[cam] is not None]
n_cams = len(valid_cams)
# 최소 공통 프레임 수
min_frames = min(frms.shape[0] for _, frms in valid_cams)

print(f"에피소드 {MULTI_EPISODE_IDX}: {n_cams}개 카메라, 최소 공통 프레임={min_frames}")

fig, axes = plt.subplots(1, n_cams, figsize=(7 * n_cams, 4.5))
if n_cams == 1:
    axes = [axes]

ims = []
texts = []
for ax, (cam, frms) in zip(axes, valid_cams):
    ax.axis('off')
    ax.set_title(cam_labels[cam], fontsize=11, fontweight='bold')
    im = ax.imshow(frms[0])
    txt = ax.text(5, 15, '', color='white', fontsize=10, fontweight='bold',
                  bbox=dict(facecolor='black', alpha=0.5, boxstyle='round,pad=0.2'))
    ims.append((im, frms))
    texts.append(txt)

fig.suptitle(f"Episode {MULTI_EPISODE_IDX} — All Cameras", fontsize=13, fontweight='bold')

def update_multi(i):
    artists = []
    for (im, frms), txt in zip(ims, texts):
        im.set_data(frms[i])
        txt.set_text(f"{i}/{min_frames-1}")
        artists.extend([im, txt])
    return artists

ani_multi = animation.FuncAnimation(fig, update_multi, frames=min_frames,
                                     interval=1000 // MULTI_FPS, blit=True)
plt.close()
HTML(ani_multi.to_jshtml())